
<a target="_blank" href="https://colab.research.google.com/github/taobrienlbl/advanced_earth_science_data_analysis/blob/fall_2025_iub/content/lessons/06_advanced_plotting/06_workalong02_mapping.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Work-along: Mapping

This workalong and exercise introduce mapping in `matplotlib`. We'll build from the figure we created in the first workalong.

In [ ]:
""" Check if this notebook is being run in Google Colab """

# if the notebook is being run in google colab, we need to install cartopy in a special way

try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False


if IN_COLAB:
  # install cartopy if it isn't already installed
  try:
    import cartopy
  except:
    # force re-installation of shapely, otherwise cartopy breaks in google colab
    # upgrade pip
    ! pip install --upgrade pip
    # remove the current version of shapely
    ! pip uninstall shapely --yes
    # install shapely and cartopy simultaneously, forcing compilation
    ! pip install --no-binary shapely cartopy
    # install cmocean too
    ! pip install cmocean

In [ ]:
""" Import libraries """


In [ ]:
""" Read in the data """
import os
import xarray as xr

# set the year we want to download
year = 2024 

# determine if we are on a UITS system through checking if the file exists in one of two expected places
path1 = f"/N/project/obrienta_startup/regcm_input_data/NNRP1/mirror/surface/air.sig995.{year}.nc"
path2 = f"/N/project/easg690/data/NNRP1/mirror/surface/air.sig995.{year}.nc"
have_local_file = False
local_path = ""
for path in [path1, path2]:
    # check if the file exists and is readable
    if os.path.exists(path) and os.access(path, os.R_OK):
        have_local_file = True
        local_path = path
        print(f"Using local file: {local_path}")
        break

# use the local file if possible
if have_local_file:
    output_file = local_path
# otherwise download the file from the NOAA server
else:
    # set the URL for the NCEP/DOE Reanalysis 2 data file
    url = f"https://psl.noaa.gov/thredds/fileServer/Datasets/ncep.reanalysis2/gaussian_grid/air.2m.gauss.{year}.nc"

    # set the name of the file we want to download to
    output_file = f"air.2m.gauss.{year}.nc"

    # download the data file
    # NOTE: the use of ! at the beginning of the line indicates that this is a shell command, not python code -- though it does use some python code.  How, why?
    # check first if the file exists; don't re-download if it does
    import os
    if not os.path.exists(output_file):
        ! curl --output {output_file} {url}

# (a side note for anyone familiar with xarray: you might ask why I don't use xarray to directly open the file from the URL (or the related OpenDAP URL)?  The reason is that it takes several minutes to open this 55 MB file, whereas directly downloading it takes only a couple seconds!)

# open the dataset using xarray
temp_ds = xr.open_dataset(output_file, chunks = -1)


""" Calculate the hottest temperature recorded in each season. """

# xarray is built on top of pandas, so we can use groupby for this
season_groups = temp_ds.groupby('time.season')

# calculate the max in each season
season_max_temp = season_groups.max()

# extract temperature (and also use 'squeeze' to remove the pesky single-item level dimension)
max_temp_xr = season_max_temp['air'].squeeze()

# pull out coordinates
seasons = max_temp_xr.season
lat = max_temp_xr.lat
lon = max_temp_xr.lon

# force the calculation
max_temp_xr.load();

## Cartopy

For mapping, we'll use `cartopy`.  Cartopy defines *projection* as the map projection on which the dataset will be drawn.  It defines the *transform* of the dataset as the projection on which the dataset exists.  They can be the same, but they don't have to be.

Let's just explore projections.

In [ ]:
""" Plot the hottest temperature recorded in each season with maplines. """
